# Notebook 03: Linear Programming Diet Optimization Prototype (PuLP)
**Capstone Project: FITCORE Gym Management & Recommendation System**

This notebook demonstrates:
1. The classic **Diet Problem** formulation (Operations Research)
2. Integer Linear Programming (ILP) using the PuLP library with CBC solver
3. Allergen exclusion and budget constraint handling
4. Constraint satisfaction verification

In [ ]:
import pandas as pd
import pulp

print(f'PuLP version: {pulp.VERSION}')
print(f'Default solver: CBC (COIN-OR Branch and Cut)')

## 1. Load the Food Dataset

The food dataset contains **52 Filipino-friendly food items** across 4 meal categories with nutritional values per serving.

In [ ]:
df = pd.read_csv('../app/data/food_dataset.csv')
print(f'Food items loaded: {len(df)}')
print(f'\nCategories:')
print(df['category'].value_counts())
print(f'\nAllergen types:')
print(df['allergens'].value_counts())
print()
df.head(10)

## 2. Problem Formulation

### Mathematical Formulation

**Decision Variables:**  
$x_i$ = number of integer servings of food item $i$, where $0 \leq x_i \leq 3$, $x_i \in \mathbb{Z}$

**Objective Function (Minimize Cost):**  
$\text{Minimize } Z = \sum_{i} (cost_i \times x_i)$

**Subject to Constraints:**
- Calorie Range: $target_{cal} - 150 \leq \sum (cal_i \times x_i) \leq target_{cal} + 150$
- Protein Minimum: $\sum (protein_i \times x_i) \geq target_{protein} \times 0.85$
- Budget Cap: $\sum (cost_i \times x_i) \leq budget$
- Serving Bounds: $0 \leq x_i \leq 3$, integer

## 3. Solve a Sample Diet Problem

**Scenario:** A gym member wants:
- Target: **2100 kcal/day**
- Protein: 30% of calories = 157g
- Budget: **₱350/day**
- Exclude: **Dairy, Nuts**

In [ ]:
# Parameters
target_cal = 2100
target_protein = target_cal * 0.30 / 4.0  # 30% of calories from protein
target_carbs = target_cal * 0.40 / 4.0    # 40% of calories from carbs
target_fat = target_cal * 0.30 / 9.0      # 30% of calories from fat
budget_limit = 350.0
excluded_allergens = ['Dairy', 'Nuts']

print(f'Target Calories : {target_cal} kcal')
print(f'Target Protein  : {target_protein:.1f}g')
print(f'Target Carbs    : {target_carbs:.1f}g')
print(f'Target Fat      : {target_fat:.1f}g')
print(f'Budget Limit    : PHP {budget_limit}')
print(f'Excluded        : {excluded_allergens}')

In [ ]:
# Step 1: Filter out allergens
df_filtered = df.copy()
for allergen in excluded_allergens:
    df_filtered = df_filtered[~df_filtered['allergens'].str.contains(allergen, case=False, na=False)]

print(f'Food items after allergen filtering: {len(df_filtered)} (removed {len(df) - len(df_filtered)} items)')
print(f'Removed items with: {excluded_allergens}')

In [ ]:
# Step 2: Formulate the LP problem
prob = pulp.LpProblem('FITCORE_DietProblem', pulp.LpMinimize)

food_ids = df_filtered['food_id'].tolist()

# Decision variables: Integer servings (0 to 3)
servings = {
    fid: pulp.LpVariable(f'srv_{fid}', lowBound=0, upBound=3, cat='Integer')
    for fid in food_ids
}

# Lookup dictionaries
cal_dict = dict(zip(df_filtered['food_id'], df_filtered['calories']))
prot_dict = dict(zip(df_filtered['food_id'], df_filtered['protein']))
carb_dict = dict(zip(df_filtered['food_id'], df_filtered['carbs']))
fat_dict = dict(zip(df_filtered['food_id'], df_filtered['fat']))
cost_dict = dict(zip(df_filtered['food_id'], df_filtered['cost']))

# Objective: Minimize total cost
prob += pulp.lpSum([servings[fid] * cost_dict[fid] for fid in food_ids]), 'Total_Cost'

# Constraint 1: Calorie range (target ± 150)
prob += pulp.lpSum([servings[fid] * cal_dict[fid] for fid in food_ids]) >= (target_cal - 150), 'Calorie_Min'
prob += pulp.lpSum([servings[fid] * cal_dict[fid] for fid in food_ids]) <= (target_cal + 150), 'Calorie_Max'

# Constraint 2: Protein >= 85% of target
prob += pulp.lpSum([servings[fid] * prot_dict[fid] for fid in food_ids]) >= (target_protein * 0.85), 'Protein_Min'

# Constraint 3: Budget limit
prob += pulp.lpSum([servings[fid] * cost_dict[fid] for fid in food_ids]) <= budget_limit, 'Budget_Limit'

print(f'LP Problem formulated:')
print(f'  Decision variables : {len(food_ids)}')
print(f'  Constraints        : {len(prob.constraints)}')
print(f'  Objective          : Minimize total cost')

In [ ]:
# Step 3: Solve
solver = pulp.PULP_CBC_CMD(msg=False)
status = prob.solve(solver)

print(f'Solver Status: {pulp.LpStatus[status]}')
print(f'Feasible: {status == pulp.constants.LpStatusOptimal}')
print(f'Optimal Cost: PHP {pulp.value(prob.objective):.2f}')

## 4. Display the Optimal Meal Plan

In [ ]:
# Collect selected food items
selected = []
for fid in food_ids:
    val = pulp.value(servings[fid])
    if val and val > 0:
        row = df_filtered[df_filtered['food_id'] == fid].iloc[0]
        selected.append({
            'Food': row['name'],
            'Category': row['category'],
            'Servings': int(val),
            'Unit': row['serving_unit'],
            'Calories': round(row['calories'] * val, 1),
            'Protein (g)': round(row['protein'] * val, 1),
            'Carbs (g)': round(row['carbs'] * val, 1),
            'Fat (g)': round(row['fat'] * val, 1),
            'Cost (PHP)': round(row['cost'] * val, 2),
        })

result_df = pd.DataFrame(selected)
print('=== OPTIMAL DAILY MEAL PLAN ===')
print(result_df.to_string(index=False))

print(f'\n--- Totals ---')
print(f"Total Calories : {result_df['Calories'].sum():.0f} kcal (target: {target_cal} ± 150)")
print(f"Total Protein  : {result_df['Protein (g)'].sum():.1f}g (target min: {target_protein * 0.85:.1f}g)")
print(f"Total Carbs    : {result_df['Carbs (g)'].sum():.1f}g")
print(f"Total Fat      : {result_df['Fat (g)'].sum():.1f}g")
print(f"Total Cost     : PHP {result_df['Cost (PHP)'].sum():.2f} (budget: PHP {budget_limit})")

## 5. Constraint Satisfaction Verification

In [ ]:
total_cal = result_df['Calories'].sum()
total_prot = result_df['Protein (g)'].sum()
total_cost = result_df['Cost (PHP)'].sum()

cal_pass = (target_cal - 150) <= total_cal <= (target_cal + 150)
prot_pass = total_prot >= target_protein * 0.85
budget_pass = total_cost <= budget_limit

print('=== CONSTRAINT SATISFACTION CHECK ===')
print(f'Calorie Range  ({target_cal-150}-{target_cal+150} kcal) : {total_cal:.0f} kcal  {"PASS" if cal_pass else "FAIL"}')
print(f'Protein Min    (>= {target_protein*0.85:.1f}g)              : {total_prot:.1f}g     {"PASS" if prot_pass else "FAIL"}')
print(f'Budget Cap     (<= PHP {budget_limit})              : PHP {total_cost:.2f}  {"PASS" if budget_pass else "FAIL"}')
print(f'\nAll constraints satisfied: {cal_pass and prot_pass and budget_pass}')

## 6. Comparison: LP vs Heuristic

The system has a **heuristic fallback** if PuLP is unavailable. Let's compare:

In [ ]:
# Heuristic: Greedily pick highest-protein item per category
categories = ['Breakfast', 'Lunch', 'Dinner', 'Snack']
heuristic_items = []
for cat in categories:
    cat_df = df_filtered[df_filtered['category'] == cat]
    if not cat_df.empty:
        best = cat_df.sort_values(by='protein', ascending=False).iloc[0]
        heuristic_items.append({
            'Food': best['name'], 'Category': cat,
            'Calories': best['calories'], 'Protein': best['protein'],
            'Cost': best['cost']
        })

heuristic_df = pd.DataFrame(heuristic_items)
h_cal = heuristic_df['Calories'].sum()
h_prot = heuristic_df['Protein'].sum()
h_cost = heuristic_df['Cost'].sum()

print('=== LP vs HEURISTIC COMPARISON ===')
print(f'{"Metric":<20} {"LP Solver":>12} {"Heuristic":>12} {"Winner":>10}')
print('-' * 56)
print(f'{"Total Calories":<20} {total_cal:>10.0f}   {h_cal:>10.0f}   {"LP" if abs(total_cal - target_cal) < abs(h_cal - target_cal) else "Heuristic":>10}')
print(f'{"Total Protein (g)":<20} {total_prot:>10.1f}   {h_prot:>10.1f}   {"LP" if total_prot > h_prot else "Heuristic":>10}')
print(f'{"Total Cost (PHP)":<20} {total_cost:>10.2f}   {h_cost:>10.2f}   {"LP" if total_cost < h_cost else "Heuristic":>10}')
print(f'\nLP guarantees optimal cost while satisfying all constraints.')
print(f'Heuristic is a fast approximation but may not meet all constraints.')

## Summary

| Property | Value |
|---|---|
| Algorithm | Integer Linear Programming (ILP) |
| Solver | PuLP CBC (COIN-OR Branch and Cut) |
| Decision Variables | Integer servings per food item (0-3) |
| Objective | Minimize total daily food cost |
| Constraints | Calorie range, protein minimum, budget cap |
| Food Dataset | 52 Filipino-friendly items with nutritional data |
| Evaluation | Feasibility, optimality, and constraint satisfaction |